# Training Analysis — Sim2Real MAPPO Traffic

This notebook loads training logs from `logs/mappo/<run_id>/` and produces:
- Reward convergence curve
- Policy / value loss curves
- Traffic performance metrics (throughput, waiting time, CO₂)
- Baseline comparison from `results/paper1_mappo/eval_tables/metrics_comparison.csv`

**Run from the project root:**
```bash
jupyter notebook notebooks/01_training_analysis.ipynb
```

In [ ]:
import sys
from pathlib import Path
from IPython.display import display

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

plt.rcParams.update({
    'font.family': 'serif',
    'font.size': 10,
    'axes.titlesize': 11,
    'axes.labelsize': 10,
    'figure.dpi': 120,
})
print('ROOT:', ROOT)

## 1. Load Training Logs

In [ ]:
RUN_ID = '20260418_215140'  # Change to your run_id
LOG_DIR = ROOT / 'logs' / 'mappo' / RUN_ID

eps_csv = LOG_DIR / 'train_episodes.csv'
upd_csv = LOG_DIR / 'train_updates.csv'

if not eps_csv.exists():
    raise FileNotFoundError(
        f'Not found: {eps_csv}\n'
        f'Run training first: python experiment/runners/train_ppo.py --sumo-cfg sumo_configs/training/sumo_config.sumocfg'
    )

eps = pd.read_csv(eps_csv)
eps['step_k'] = eps['global_step'] / 1_000

upd = pd.read_csv(upd_csv)
upd['step_k'] = upd['global_step'] / 1_000

print(f'Episodes : {len(eps):,}')
print(f'Updates  : {len(upd):,}')
print(f'Steps    : {int(eps.global_step.min()):,} – {int(eps.global_step.max()):,}')
eps.tail()

## 2. Reward Convergence

In [ ]:
WINDOW = 50

fig, ax = plt.subplots(figsize=(9, 4))

raw   = eps['mean_reward']
roll  = raw.rolling(WINDOW, min_periods=1).mean()
std   = raw.rolling(WINDOW, min_periods=1).std().fillna(0)
ep    = eps['episode'].values

ax.fill_between(ep, (roll - std).values, (roll + std).values, alpha=0.25, color='steelblue')
ax.plot(ep, raw.values,  color='lightgray', linewidth=0.4, alpha=0.6, label='Raw')
ax.plot(ep, roll.values, color='steelblue', linewidth=1.5, label=f'Rolling avg ({WINDOW} ep)')

tail_mean = roll.iloc[int(len(roll) * 0.8):].mean()
ax.axhline(tail_mean, linestyle='--', color='steelblue', linewidth=0.9, alpha=0.7)
ax.text(ep[-1] * 0.98, tail_mean + 0.5, f'Converged: {tail_mean:.2f}', ha='right', fontsize=8)

ax.set_xlabel('Episode')
ax.set_ylabel('Mean Reward')
ax.set_title('Reward Convergence')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 3. Traffic Performance Metrics

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

metrics = [
    ('throughput',         'Throughput (veh/ep)',  'darkorange'),
    ('avg_waiting_proxy',  'Avg Waiting Time (s)', 'crimson'),
    ('queue_total_proxy',  'Total Queue (veh)',    'teal'),
]

for ax, (col, label, color) in zip(axes, metrics):
    if col not in eps.columns:
        ax.set_title(f'{label}\n(not in log)')
        continue
    roll_m = eps[col].rolling(WINDOW, min_periods=1).mean()
    roll_s = eps[col].rolling(WINDOW, min_periods=1).std().fillna(0)
    ax.fill_between(ep, (roll_m - roll_s).values, (roll_m + roll_s).values,
                    alpha=0.2, color=color)
    ax.plot(ep, roll_m.values, color=color, linewidth=1.5)
    ax.set_xlabel('Episode')
    ax.set_ylabel(label)
    ax.set_title(label)
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 4. Policy & Value Losses

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
W2 = 200

loss_metrics = [
    ('policy_loss', 'Policy Loss (PPO clip)', 'crimson'),
    ('value_loss',  'Value Loss',             'purple'),
    ('entropy',     'Policy Entropy H[π]',    'teal'),
]

for ax, (col, label, color) in zip(axes, loss_metrics):
    s = upd['step_k'].values
    roll_m = upd[col].rolling(W2, min_periods=1).mean()
    ax.plot(s, upd[col].values, color=color, alpha=0.15, linewidth=0.4)
    ax.plot(s, roll_m.values,   color=color, linewidth=1.5, label=f'Rolling avg ({W2})')
    ax.set_xlabel('Global Step (×10³)')
    ax.set_ylabel(label)
    ax.set_title(label)
    ax.legend(fontsize=8)
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 5. Baseline Comparison Table

In [ ]:
metrics_csv = ROOT / 'results' / 'paper1_mappo' / 'eval_tables' / 'metrics_comparison.csv'

if metrics_csv.exists():
    df = pd.read_csv(metrics_csv, sep=';')
    display(df.style.highlight_min(axis=1, subset=['No PPO (Fixed)'], color='#ffd0d0')
                    .highlight_max(axis=1, subset=['PPO (AI)'],       color='#d0ffd0')
                    .set_caption('Baseline vs MAPPO Comparison'))
else:
    print(f'File not found: {metrics_csv}')

## 6. Summary Statistics

In [ ]:
last_20pct = int(len(eps) * 0.8)
converged  = eps.iloc[last_20pct:]

summary = {
    'Total episodes': len(eps),
    'Total updates': len(upd),
    'Final mean reward (last 20%)': f"{converged['mean_reward'].mean():.3f} ± {converged['mean_reward'].std():.3f}",
}
for col in ['throughput', 'avg_waiting_proxy', 'queue_total_proxy']:
    if col in converged.columns:
        summary[f'Converged {col}'] = f"{converged[col].mean():.2f} ± {converged[col].std():.2f}"

for k, v in summary.items():
    print(f'{k:<45} {v}')